In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#путь к папке с корпусом (лемматизированный корпус)
path = '/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt'
dirs = os.listdir(path)

NotADirectoryError: [Errno 20] Not a directory: '/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt'

In [ ]:
i = 0
# словарь с именем файла и меткой класса (например, название рубрики или значение тональности)
dict_score = {}

In [ ]:
for file in dirs:
    #есть нетекстовые файлы
    if (file[-4:] == '.txt'):
        with open(path + '/' + file, 'r', encoding='utf-8') as readfile:
            for line in readfile:
                list_words = line.split()
                # последним токеном в файле идёт оценка = для текстов отзывов
                dict_score[file] = list_words[len(list_words)-1]
                # формируем файл для анализа
                with open('reviews_for_work.txt', 'a', encoding='utf-8') as wfile:
                    # первым токеном в файле идёт номер файла, он не нужен
                    wfile.write(' '.join(list_words[1:len(list_words) - 1]) + '\n')
                    i += 1

In [ ]:
print(f'files: {i}')

In [ ]:
print(f'dict_score: {len(dict_score)}')

In [ ]:
import re
import math
import pandas as pd
import nltk

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as file: #файл с текстом
    f = file.read()

In [ ]:
text_words_2 = re.findall(r'[а-яёА-ЯЁ]+', f)

text_words_1 = [] #список с декапитализированными словами
for i in text_words_2:
    k = i.lower()
    text_words_1.append(k)

text_words_2

['Дополнительные',
 'функции',
 'новостных',
 'Б',
 'Р',
 'Певзнер',
 'СВЕТЛОЙ',
 'ПАМЯТИ',
 'МОЕГО',
 'ДРУГА',
 'И',
 'ДРУЖЕСТВЕННОГО',
 'ОППОНЕНТА',
 'ВАЛЕРИЯ',
 'РУБАШКИНА',
 'Выделение',
 'из',
 'новостного',
 'информационного',
 'потока',
 'релевантных',
 'сообщений',
 'Разделение',
 'входящего',
 'потока',
 'текстовых',
 'сообщений',
 'на',
 'две',
 'части',
 'то',
 'что',
 'может',
 'быть',
 'интересно',
 'для',
 'пользователя',
 'и',
 'то',
 'что',
 'ему',
 'мало',
 'интересно',
 'Эта',
 'функция',
 'на',
 'самом',
 'деле',
 'выделяет',
 'релевантную',
 'информацию',
 'для',
 'данного',
 'пользователя',
 'из',
 'потока',
 'Известно',
 'что',
 'поток',
 'состоит',
 'из',
 'текстовых',
 'сообщений',
 'относящихся',
 'к',
 'различным',
 'предметам',
 'и',
 'событиям',
 'Сегодня',
 'пользователь',
 'читатель',
 'должен',
 'просматривать',
 'все',
 'текстовые',
 'сообщения',
 'вернее',
 'заголовки',
 'потока',
 'для',
 'того',
 'чтобы',
 'выбрать',
 'и',
 'открыть',
 'нужные',
 'Я',

In [ ]:
text_words_1

['дополнительные',
 'функции',
 'новостных',
 'б',
 'р',
 'певзнер',
 'светлой',
 'памяти',
 'моего',
 'друга',
 'и',
 'дружественного',
 'оппонента',
 'валерия',
 'рубашкина',
 'выделение',
 'из',
 'новостного',
 'информационного',
 'потока',
 'релевантных',
 'сообщений',
 'разделение',
 'входящего',
 'потока',
 'текстовых',
 'сообщений',
 'на',
 'две',
 'части',
 'то',
 'что',
 'может',
 'быть',
 'интересно',
 'для',
 'пользователя',
 'и',
 'то',
 'что',
 'ему',
 'мало',
 'интересно',
 'эта',
 'функция',
 'на',
 'самом',
 'деле',
 'выделяет',
 'релевантную',
 'информацию',
 'для',
 'данного',
 'пользователя',
 'из',
 'потока',
 'известно',
 'что',
 'поток',
 'состоит',
 'из',
 'текстовых',
 'сообщений',
 'относящихся',
 'к',
 'различным',
 'предметам',
 'и',
 'событиям',
 'сегодня',
 'пользователь',
 'читатель',
 'должен',
 'просматривать',
 'все',
 'текстовые',
 'сообщения',
 'вернее',
 'заголовки',
 'потока',
 'для',
 'того',
 'чтобы',
 'выбрать',
 'и',
 'открыть',
 'нужные',
 'я',

In [ ]:
with open('/content/drive/MyDrive/СемАн/swl.txt', 'r', encoding='utf8') as swl: #NEED STOP-WORDS LIST
    swlist = []  # список стоп-слов из файла
    for i in swl:
        i = i.replace('\n','')
        swlist.append(i)
        # print(swlist)

len(swlist)

941

In [ ]:
!pip install pymorphy3
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [ ]:
text_words = [] #список без стоп-слов
for i in text_words_1:
    i = morph.parse(i)[0].normal_form
    if i not in swlist:
        text_words.append(i)

text_words

In [ ]:
len(text_words)

419

In [ ]:
frequency={}
for word in text_words:  # рассчитываем частоты слов
    count = frequency.get(word,0)  # создаём словарь СЛОВО и ЧАСТОТА
    frequency[word] = count + 1  # увеличиваем ЧАСТОТУ при считывании одинаковых лемм слова

frequency

{'дополнительный': 1,
 'функция': 3,
 'новостной': 5,
 'р': 1,
 'певзнер': 1,
 'светлый': 1,
 'память': 1,
 'друг': 1,
 'дружественный': 1,
 'оппонент': 1,
 'валерий': 1,
 'рубашкин': 1,
 'выделение': 1,
 'информационный': 7,
 'поток': 17,
 'релевантный': 3,
 'сообщение': 23,
 'разделение': 1,
 'входящий': 3,
 'текстовый': 4,
 'часть': 2,
 'интересно': 2,
 'пользователь': 15,
 'дело': 1,
 'выделять': 1,
 'информация': 2,
 'известно': 3,
 'состоять': 3,
 'относиться': 1,
 'различный': 2,
 'предмет': 1,
 'событие': 6,
 'сегодня': 1,
 'читатель': 7,
 'должный': 2,
 'просматривать': 1,
 'верный': 1,
 'заголовок': 1,
 'выбрать': 1,
 'открыть': 1,
 'нужный': 1,
 'разработать': 3,
 'алгоритм': 8,
 'деление': 1,
 'тип': 1,
 'интересный': 1,
 'ины': 1,
 'блок': 7,
 'создать': 2,
 'кластер': 10,
 'включать': 1,
 'входной': 4,
 'регистрация': 1,
 'деятельность': 2,
 'просмотр': 1,
 'новый': 2,
 'следить': 1,
 'специальный': 1,
 'программа': 1,
 'супервизер': 1,
 'фиксировать': 1,
 'читаться': 1,


In [ ]:
freq = sorted(frequency.items(), key=lambda x:x[1], reverse = True) #создаём список, в котором пары СЛОВО и ЧАСТОТА упорядочены по убыванию
length = len(text_words)

print(length)
print(freq)

419
[('сообщение', 23), ('поток', 17), ('пользователь', 15), ('кластер', 10), ('алгоритм', 8), ('информационный', 7), ('читатель', 7), ('блок', 7), ('реклама', 7), ('событие', 6), ('являться', 6), ('профиль', 6), ('новостной', 5), ('пользовательский', 5), ('запрос', 5), ('разный', 5), ('время', 5), ('объект', 5), ('текстовый', 4), ('входной', 4), ('подобный', 4), ('база', 4), ('данные', 4), ('функция', 3), ('релевантный', 3), ('входящий', 3), ('известно', 3), ('состоять', 3), ('разработать', 3), ('расстояние', 3), ('сторона', 3), ('иметь', 3), ('сайт', 3), ('адресный', 3), ('описывать', 3), ('описать', 3), ('помощь', 3), ('сеть', 3), ('часть', 2), ('интересно', 2), ('информация', 2), ('различный', 2), ('должный', 2), ('создать', 2), ('деятельность', 2), ('новый', 2), ('представлять', 2), ('статистический', 2), ('критерий', 2), ('источник', 2), ('процедура', 2), ('проект', 2), ('страница', 2), ('появляться', 2), ('идея', 2), ('связь', 2), ('поступать', 2), ('собственный', 2), ('имя', 2)

In [ ]:
n_list=[]
for words in freq:  # проверяем каждую пару СЛОВО и ЧАСТОТА в отсортированном тексте
    fr_lst = list(words) # превращаем пары в круглых скобках, созданные анонимной функцией LAMBDA, в список из двух элементов - СЛОВО и ЧАСТОТА
    n_list.append(fr_lst)

n_list

[['сообщение', 23],
 ['поток', 17],
 ['пользователь', 15],
 ['кластер', 10],
 ['алгоритм', 8],
 ['информационный', 7],
 ['читатель', 7],
 ['блок', 7],
 ['реклама', 7],
 ['событие', 6],
 ['являться', 6],
 ['профиль', 6],
 ['новостной', 5],
 ['пользовательский', 5],
 ['запрос', 5],
 ['разный', 5],
 ['время', 5],
 ['объект', 5],
 ['текстовый', 4],
 ['входной', 4],
 ['подобный', 4],
 ['база', 4],
 ['данные', 4],
 ['функция', 3],
 ['релевантный', 3],
 ['входящий', 3],
 ['известно', 3],
 ['состоять', 3],
 ['разработать', 3],
 ['расстояние', 3],
 ['сторона', 3],
 ['иметь', 3],
 ['сайт', 3],
 ['адресный', 3],
 ['описывать', 3],
 ['описать', 3],
 ['помощь', 3],
 ['сеть', 3],
 ['часть', 2],
 ['интересно', 2],
 ['информация', 2],
 ['различный', 2],
 ['должный', 2],
 ['создать', 2],
 ['деятельность', 2],
 ['новый', 2],
 ['представлять', 2],
 ['статистический', 2],
 ['критерий', 2],
 ['источник', 2],
 ['процедура', 2],
 ['проект', 2],
 ['страница', 2],
 ['появляться', 2],
 ['идея', 2],
 ['связь', 2

In [ ]:
k = 1
for i in n_list:
    logfr = math.log(int(i[1]),10)
    logrank = math.log(k, 10)  # берём десятичные логарифмы для ранга и для частоты
    i.append(k)
    i.append(logfr)
    i.append(logrank)
    k+=1

In [ ]:
n_list

[['сообщение', 23, 1, 1.3617278360175928, 0.0],
 ['поток', 17, 2, 1.2304489213782739, 0.30102999566398114],
 ['пользователь', 15, 3, 1.1760912590556811, 0.47712125471966244],
 ['кластер', 10, 4, 1.0, 0.6020599913279623],
 ['алгоритм', 8, 5, 0.9030899869919434, 0.6989700043360187],
 ['информационный', 7, 6, 0.8450980400142567, 0.7781512503836435],
 ['читатель', 7, 7, 0.8450980400142567, 0.8450980400142567],
 ['блок', 7, 8, 0.8450980400142567, 0.9030899869919434],
 ['реклама', 7, 9, 0.8450980400142567, 0.9542425094393249],
 ['событие', 6, 10, 0.7781512503836435, 1.0],
 ['являться', 6, 11, 0.7781512503836435, 1.041392685158225],
 ['профиль', 6, 12, 0.7781512503836435, 1.0791812460476247],
 ['новостной', 5, 13, 0.6989700043360187, 1.1139433523068367],
 ['пользовательский', 5, 14, 0.6989700043360187, 1.1461280356782377],
 ['запрос', 5, 15, 0.6989700043360187, 1.1760912590556811],
 ['разный', 5, 16, 0.6989700043360187, 1.2041199826559246],
 ['время', 5, 17, 0.6989700043360187, 1.230448921378

In [ ]:
frame = pd.DataFrame(n_list, columns=['Words','Frequencies','Ranks', 'Logarithm of Frequencies', 'Logarithm of ranks']) # собираем фрейм
frame.to_csv('Frequency Dictionary.csv',encoding='utf-8', sep=';', index=False) #экспортируем в файл

In [ ]:
frame

,Words,Frequencies,Ranks,Logarithm of Frequencies,Logarithm of ranks
0,сообщение,23,1,1.361728,0.000000
1,поток,17,2,1.230449,0.301030
2,пользователь,15,3,1.176091,0.477121
3,кластер,10,4,1.000000,0.602060
4,алгоритм,8,5,0.903090,0.698970
...,...,...,...,...,...
212,яркий,1,213,0.000000,2.328380
213,пример,1,214,0.000000,2.330414
214,система,1,215,0.000000,2.332438
215,машинный,1,216,0.000000,2.334454


In [ ]:
bigrams = nltk.collocations.BigramAssocMeasures()
trigrams = nltk.collocations.TrigramAssocMeasures()

bigramFinder = nltk.collocations.BigramCollocationFinder.from_words(text_words)
trigramFinder = nltk.collocations.TrigramCollocationFinder.from_words(text_words)

# bigrams
bigram_freq = bigramFinder.ngram_fd.items()
bigramFreqTable = pd.DataFrame(list(bigram_freq), columns=['bigram','freq']).sort_values(by='freq', ascending=False)
bigramFreqTable.to_csv('Bigrams.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

# trigrams
trigram_freq = trigramFinder.ngram_fd.items()
trigramFreqTable = pd.DataFrame(list(trigram_freq), columns=['trigram','freq']).sort_values(by='freq', ascending=False)
trigramFreqTable.to_csv('Trigrams.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

In [ ]:
bigram_freq

dict_items([(('дополнительный', 'функция'), 1), (('функция', 'новостной'), 1), (('новостной', 'р'), 1), (('р', 'певзнер'), 1), (('певзнер', 'светлый'), 1), (('светлый', 'память'), 1), (('память', 'друг'), 1), (('друг', 'дружественный'), 1), (('дружественный', 'оппонент'), 1), (('оппонент', 'валерий'), 1), (('валерий', 'рубашкин'), 1), (('рубашкин', 'выделение'), 1), (('выделение', 'новостной'), 1), (('новостной', 'информационный'), 1), (('информационный', 'поток'), 1), (('поток', 'релевантный'), 1), (('релевантный', 'сообщение'), 2), (('сообщение', 'разделение'), 1), (('разделение', 'входящий'), 1), (('входящий', 'поток'), 1), (('поток', 'текстовый'), 2), (('текстовый', 'сообщение'), 4), (('сообщение', 'часть'), 1), (('часть', 'интересно'), 1), (('интересно', 'пользователь'), 1), (('пользователь', 'интересно'), 1), (('интересно', 'функция'), 1), (('функция', 'дело'), 1), (('дело', 'выделять'), 1), (('выделять', 'релевантный'), 1), (('релевантный', 'информация'), 1), (('информация', 'по

In [ ]:
# t-test
bigramTtable = pd.DataFrame(list(bigramFinder.score_ngrams(bigrams.student_t)), columns=['bigram','T-test']).sort_values(by='T-test', ascending=False)
trigramTtable = pd.DataFrame(list(trigramFinder.score_ngrams(trigrams.student_t)), columns=['trigram','T-test']).sort_values(by='T-test', ascending=False)
bigramTtable = bigramTtable.head(20)
trigramTtable = trigramTtable.head(20) #вычленяем первые 20 лучших би- и триграмм
bigramTtable.to_csv('Bigrams-T-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл
trigramTtable.to_csv('Trigrams-T-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

# chi-squared
bigramCHItable = pd.DataFrame(list(bigramFinder.score_ngrams(bigrams.chi_sq)), columns=['bigram','Chi-squared']).sort_values(by='Chi-squared', ascending=False)
trigramCHItable = pd.DataFrame(list(trigramFinder.score_ngrams(trigrams.chi_sq)), columns=['trigram','Chi-squared']).sort_values(by='Chi-squared', ascending=False)
bigramCHItable = bigramCHItable.head(20)
trigramCHItable = trigramCHItable.head(20) #вычленяем первые 20 лучших би- и триграмм
bigramCHItable.to_csv('Bigrams-CHI-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл
trigramCHItable.to_csv('Trigrams-CHI-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

# log-likelihood
bigramlikelihoodtable = pd.DataFrame(list(bigramFinder.score_ngrams(bigrams.likelihood_ratio)), columns=['bigram','Log-Likelihood']).sort_values(by='Log-Likelihood', ascending=False)
trigramlikelihoodtable = pd.DataFrame(list(trigramFinder.score_ngrams(trigrams.likelihood_ratio)), columns=['trigram','Log-Likelihood']).sort_values(by='Log-Likelihood', ascending=False)
bigramlikelihoodtable = bigramlikelihoodtable.head(20)
trigramlikelihoodtable = trigramlikelihoodtable.head(20) #вычленяем первые 20 лучших би- и триграмм
bigramlikelihoodtable.to_csv('Bigrams-likelihood-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл
trigramlikelihoodtable.to_csv('Trigrams-likelihood-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

# pmi
bigramPMItable = pd.DataFrame(list(bigramFinder.score_ngrams(bigrams.pmi)), columns=['bigram','PMI']).sort_values(by='PMI', ascending=False)
trigramPMItable = pd.DataFrame(list(trigramFinder.score_ngrams(trigrams.pmi)), columns=['trigram','PMI']).sort_values(by='PMI', ascending=False)
bigramPMItable = bigramPMItable.head(20)
trigramPMItable = trigramPMItable.head(20) #вычленяем первые 20 лучших би- и триграмм
bigramPMItable.to_csv('Bigrams-PMI-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл
trigramPMItable.to_csv('Trigrams-PMI-test-20.csv',encoding='utf8', sep=';', index=False) #экспортируем в файл

In [ ]:
#RAKE

In [ ]:
import pymorphy3 as pm
import codecs
import string

In [ ]:
m = pm.MorphAnalyzer()

In [ ]:
words = []
words2 = ""

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as f: #прописываем путь к корпусу (папка или файл)
    text = f.read()
    words = text.split()
    for i in range(len(words) - 1):
        if m.parse(words[i])[0].tag.POS == "ADVB" or m.parse(words[i])[0].tag.POS == "VERB" or m.parse(words[i][:-1])[0].tag.POS == "VERB" or m.parse(words[i])[0].tag.POS == "PRTS"or m.parse(words[i])[0].tag.POS == "INFN"or m.parse(words[i])[0].tag.POS == "COMP" or m.parse(words[i])[0].tag.POS == "ADJS" or m.parse(words[i])[0].tag.POS == "GRND"or m.parse(words[i])[0].tag.POS == "CONJ":
            words2 += " | " + words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i])[0].tag.case == ("gent" or "accs")):
            words2 += words[i] + " | "
        elif m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i + 1])[0].tag.POS == ("NOUN" or "ADJF") and m.parse(words[i])[0].tag.case != m.parse(words[i + 1])[0].tag.case and m.parse(words[i + 1])[0].tag.case != "gent":
            words2 += words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i])[0].tag.case != ("nomn" or "accs") and m.parse(words[i + 1])[0].tag.POS == "NOUN" and m.parse(words[i + 1])[0].tag.case == ("nomn" or "accs")):
            words2 += words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == None and m.parse(words[i][:-1])[0].tag.POS == None and m.parse(words[i][1:])[0].tag.POS == None):
            words2 += " | " + words[i] + " | "
        elif (m.parse(words[i])[0].tag.POS == "NOUN" and m.parse(words[i - 1])[0].tag.POS == "ADJF" or m.parse(words[i - 1])[0].tag.POS == "PRTF") and m.parse(words[i])[0].tag.case == m.parse(words[i + 1])[0].tag.case:
            words2 += words[i] + " | "
        else:
            words2 += words[i] + " "

In [ ]:
words2

'Дополнительные функции новостных  | web |  | sites |  | Б.Р. | Певзнер  | mtpl.boris@ |  | gmail.com | СВЕТЛОЙ ПАМЯТИ | МОЕГО ДРУГА |  | И | ДРУЖЕСТВЕННОГО ОППОНЕНТА | ВАЛЕРИЯ | РУБАШКИНА |  | 1. | Выделение из новостного информационного потока | релевантных  | сообщений | Разделение входящего потока | текстовых  | сообщений | на две части: то,  | что |  | может |  | быть |  | интересно | для  | пользователя |  | и | то,  | что |  | ему | мало интересно. Эта функция на самом деле  | выделяет |  | релевантную | информацию для данного  | пользователя | из потока. Известно,  | что | поток  | состоит | из текстовых сообщений, относящихся к различным предметам  | и | событиям.  | Сегодня |  | пользователь | (читатель)  | должен |  | просматривать | все текстовые  | сообщения |  | (вернее | заголовки) потока | для того,  | чтобы |  | выбрать |  | и |  | открыть | нужные. Я  | разработал |  | алгоритм | деления | потока | текстовых  | сообщений | на два типа: те, которые  | могут |  | быть |

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'w', encoding='utf-8') as f2:
    f2.write(words2)

In [ ]:
from __future__ import division
import operator
import nltk

In [ ]:
punct = open("/content/drive/MyDrive/СемАн/stopwords/stopwords/punct.txt", 'r', encoding='utf-8').read()

In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
punct_list = nltk.word_tokenize(punct)

In [ ]:
stop = open("/content/drive/MyDrive/СемАн/stopwords/stopwords/stop.txt", 'r', encoding='utf-8').read()

In [ ]:
pronouns = open("/content/drive/MyDrive/СемАн/stopwords/stopwords/pronouns.txt", 'r', encoding='utf-8').read()

In [ ]:
preps = open("/content/drive/MyDrive/СемАн/stopwords/stopwords/preps.txt", 'r', encoding='utf-8').read()

In [ ]:
preps

'а-ля\nбез\nбезо\nблагодаря\nблиз\nв\nвблизи\nввиду\nвглубь\nвдогон\nвдоль\nвзамен\nвключая\nвкруг\nвместо\nвне\nвнизу\nвнутри\nвнутрь\nво\nво имя\nвовнутрь\nвозле\nвокруг\nвопреки\nвослед\nвпереди\nвразрез\nвроде\nвслед\nвследствие\nдля\nдля-ради\nдо\nза\nзаместо\nиз\nиз-за\nиз-под\nизнутри\nизо\nк\nкасательно\nко\nкроме\nкругом\nмеж\nмежду\nмимо\nна\nнаверху\nнавроде\nнавстречу\nнад\nнадо\nназад\nнакануне\nнаперекор\nнаперерез\nнаподобие\nнапротив\nнасупротив\nнасчёт\nниже\nо\nоб\nобо\nобок\nоколо\nокрест\nокромя\nокруг\nопосля\nот\nотносительно\nото\nперед\nпередо\nпо\nпо-за\nпо-над\nпо-под\nповерх\nпод\nподле\nподо\nподобно\nпозади\nпозднее\nпомимо\nпоперёд\nпоперёк\nпорядка\nпосередине\nпосередь\nпосле\nпосреди\nпосредине\nпосредством\nпред\nпредо\nпрежде\nпри\nпро\nпротив\nпутём\nради\nс\nсверх\nсверху\nсвыше\nсзади\nсквозь\nскрозь\nснизу\nсо\nсогласно\nспустя\nсреди\nсредь\nсродни\nсупротив\nчерез\nчрез\n'

In [ ]:
mystop = open("/content/drive/MyDrive/СемАн/stopwords/stopwords/stop.txt", 'r', encoding='utf-8').read()

In [ ]:
#nltk.word_tokenize(preps.decode("utf8")) + nltk.word_tokenize(mystop.decode("utf8"))
nltk.word_tokenize(preps) + nltk.word_tokenize(mystop)

['а-ля',
 'без',
 'безо',
 'благодаря',
 'близ',
 'в',
 'вблизи',
 'ввиду',
 'вглубь',
 'вдогон',
 'вдоль',
 'взамен',
 'включая',
 'вкруг',
 'вместо',
 'вне',
 'внизу',
 'внутри',
 'внутрь',
 'во',
 'во',
 'имя',
 'вовнутрь',
 'возле',
 'вокруг',
 'вопреки',
 'вослед',
 'впереди',
 'вразрез',
 'вроде',
 'вслед',
 'вследствие',
 'для',
 'для-ради',
 'до',
 'за',
 'заместо',
 'из',
 'из-за',
 'из-под',
 'изнутри',
 'изо',
 'к',
 'касательно',
 'ко',
 'кроме',
 'кругом',
 'меж',
 'между',
 'мимо',
 'на',
 'наверху',
 'навроде',
 'навстречу',
 'над',
 'надо',
 'назад',
 'накануне',
 'наперекор',
 'наперерез',
 'наподобие',
 'напротив',
 'насупротив',
 'насчёт',
 'ниже',
 'о',
 'об',
 'обо',
 'обок',
 'около',
 'окрест',
 'окромя',
 'округ',
 'опосля',
 'от',
 'относительно',
 'ото',
 'перед',
 'передо',
 'по',
 'по-за',
 'по-над',
 'по-под',
 'поверх',
 'под',
 'подле',
 'подо',
 'подобно',
 'позади',
 'позднее',
 'помимо',
 'поперёд',
 'поперёк',
 'порядка',
 'посередине',
 'посередь',
 

In [ ]:
stop_list = nltk.word_tokenize(stop) + nltk.word_tokenize(pronouns) + nltk.word_tokenize(preps) + nltk.word_tokenize(mystop)

In [ ]:
def isPunct(word):
  return len(word) == 1 and (word in string.punctuation or word in punct_list)

In [ ]:
def isNumeric(word):
  try:
    float(word) if '.' in word else int(word)
    return True
  except ValueError:
    return False

In [ ]:
def isInitial(word):
  return len(word) == 2 and word[1] == "."

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
class RakeKeywordExtractor:

  def __init__(self):
    self.stopwords = set(nltk.corpus.stopwords.words())
    self.top_fraction = 1 # consider top third candidate keywords by score

  def _generate_candidate_keywords(self, sentences):
    phrase_list = []
    for sentence in sentences:
      words = map(lambda x: "|" if x in self.stopwords or x in stop_list or isNumeric(x) or isInitial(x) else x,
        nltk.word_tokenize(sentence.lower()))
      phrase = []
      for word in words:
        if word == "|" or isPunct(word):
          if len(phrase) > 0:
            phrase_list.append(phrase)
            phrase = []
        else:
          phrase.append(word)
    return phrase_list
## СТОП!!
  def _calculate_word_scores(self, phrase_list):
    word_freq = nltk.FreqDist()
    word_degree = nltk.FreqDist()
    for phrase in phrase_list:
      #degree = len(filter(lambda x: not isNumeric(x), phrase)) - 1
      degree = len(list(filter(lambda x: not isNumeric(x), phrase))) - 1
      for word in phrase:
        word_freq[word] += 1 ## word_fd.inc(word.lower()) with word_fd[word.lower()] += 1
        word_degree[word] += degree # other words
    for word in word_freq.keys():
      word_degree[word] = word_degree[word] + word_freq[word] # itself
    # word score = deg(w) / freq(w)
    word_scores = {}
    for word in word_freq.keys():
      word_scores[word] = word_degree[word] / word_freq[word]
    return word_scores

  def _calculate_phrase_scores(self, phrase_list, word_scores):
    phrase_scores = {}
    for phrase in phrase_list:
      phrase_score = 0
      for word in phrase:
        phrase_score += word_scores[word]
      phrase_scores[" ".join(phrase)] = phrase_score
    return phrase_scores

  def extract(self, text, incl_scores=False):
    sentences = nltk.sent_tokenize(text)
    phrase_list = self._generate_candidate_keywords(sentences)
    word_scores = self._calculate_word_scores(phrase_list)
    phrase_scores = self._calculate_phrase_scores(
      phrase_list, word_scores)
    #sorted_phrase_scores = sorted(phrase_scores.iteritems(),
    sorted_phrase_scores = sorted(phrase_scores.items(),
      key=operator.itemgetter(1), reverse=True)
    n_phrases = len(sorted_phrase_scores)
    if incl_scores:
      return sorted_phrase_scores[0:int(n_phrases/self.top_fraction)]
    else:
      return map(lambda x: x[0],
        sorted_phrase_scores[0:int(n_phrases/self.top_fraction)])

In [ ]:
rake = RakeKeywordExtractor()

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
    txt = f.read()

In [ ]:
keywords = rake.extract(txt, incl_scores=True)

In [ ]:
for eachkwrd in keywords:
    print(eachkwrd[0], eachkwrd[1])

каждым новым сообщением входного потока 20.0
помощью различных семантических средств 14.666666666666666
созданный самим пользователем 9.0
статистический критерий расстояния 9.0
близкими информационными потребностями 9.0
ключевыми словами запроса 9.0
картину описанных объектов 9.0
создание социальной сети 9.0
основе информационной деятельности 9.0
реализация этих алгоритмов 9.0
эффективность новостных агенств 9.0
качестве составного элемента 9.0
помощью собственных имен 8.666666666666666
дополнительные функции новостных 8.5
информационная часть алгоритма 8.5
двух информационных блоков 8.5
московского гуманитарного университета 8.5
новостных сайтах запросов 8.333333333333334
разделение входящего потока 8.0
собой статистический критерий 8.0
следствием фундаментальной идеи 8.0
просмотре новых сообщений 7.6
новостного информационного потока 7.5
адресная реклама сети 7.5
входного потока 5.0
ядро алгоритма 4.5
адресная реклама 4.5
пользовательских блоков 4.5
функции поиска 4.5
брауншвейгского

In [ ]:
keywords

[('каждым новым сообщением входного потока', 20.0),
 ('помощью различных семантических средств', 14.666666666666666),
 ('созданный самим пользователем', 9.0),
 ('статистический критерий расстояния', 9.0),
 ('близкими информационными потребностями', 9.0),
 ('ключевыми словами запроса', 9.0),
 ('картину описанных объектов', 9.0),
 ('создание социальной сети', 9.0),
 ('основе информационной деятельности', 9.0),
 ('реализация этих алгоритмов', 9.0),
 ('эффективность новостных агенств', 9.0),
 ('качестве составного элемента', 9.0),
 ('помощью собственных имен', 8.666666666666666),
 ('дополнительные функции новостных', 8.5),
 ('информационная часть алгоритма', 8.5),
 ('двух информационных блоков', 8.5),
 ('московского гуманитарного университета', 8.5),
 ('новостных сайтах запросов', 8.333333333333334),
 ('разделение входящего потока', 8.0),
 ('собой статистический критерий', 8.0),
 ('следствием фундаментальной идеи', 8.0),
 ('просмотре новых сообщений', 7.6),
 ('новостного информационного по

In [ ]:
with open('rake_keywords.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [ ]:
#KeyBERT
!pip install keybert
import keybert

In [ ]:
from keybert import KeyBERT

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
    doc = f.read()

In [ ]:
kw_model = KeyBERT()

In [ ]:
keywords = kw_model.extract_keywords(doc)

In [ ]:
keywords

[('информации', 0.4102),
 ('информационного', 0.3995),
 ('информационных', 0.3924),
 ('расстояния', 0.3912),
 ('информацию', 0.3876)]

In [ ]:
keywords = kw_model.extract_keywords(doc, top_n=20)
keywords

[('информации', 0.4102),
 ('информационного', 0.3995),
 ('информационных', 0.3924),
 ('расстояния', 0.3912),
 ('информацию', 0.3876),
 ('информационной', 0.3859),
 ('информационными', 0.3844),
 ('относящихся', 0.38),
 ('интернеткомпаний', 0.3782),
 ('информационный', 0.3765),
 ('информационная', 0.3672),
 ('сообщениями', 0.3637),
 ('сообщениях', 0.3598),
 ('описанных', 0.3547),
 ('источником', 0.3476),
 ('профилем', 0.3428),
 ('повысит', 0.3391),
 ('описанные', 0.338),
 ('профилей', 0.3379),
 ('описывают', 0.3377)]

In [ ]:
with open('KeyBERT_top_20.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [ ]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 2), top_n=20)
keywords

[('описаны сообщениях', 0.4643),
 ('сообщения информационная', 0.4577),
 ('новостного информационного', 0.4499),
 ('сообщения описываются', 0.4454),
 ('основе информационной', 0.4371),
 ('информационного потока', 0.4366),
 ('информационной деятельности', 0.4364),
 ('критерий расстояния', 0.4234),
 ('входящие сообщения', 0.4223),
 ('объединение профилей', 0.4221),
 ('сообщений относящихся', 0.4202),
 ('информационными потребностями', 0.4182),
 ('иныe сообщения', 0.4148),
 ('информационного блока', 0.4134),
 ('информации', 0.4102),
 ('новостных web', 0.4076),
 ('сообщениями поступающими', 0.4073),
 ('профилей входящих', 0.4061),
 ('относящихся различным', 0.4052),
 ('блок информации', 0.4049)]

In [ ]:
with open('KeyBERT_top_20_bigram.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [ ]:
keywords = kw_model.extract_keywords(doc, keyphrase_ngram_range=(1, 3), top_n=20)
keywords

[('иныe сообщения информационная', 0.4919),
 ('новостного информационного потока', 0.4831),
 ('подобные сообщения описываются', 0.4765),
 ('описаны сообщениях помощью', 0.4755),
 ('объединение профилей входящих', 0.4736),
 ('основе информационной деятельности', 0.4721),
 ('функции новостных web', 0.4666),
 ('описаны сообщениях', 0.4643),
 ('сообщения описываются одними', 0.4627),
 ('информационного потока релевантных', 0.4615),
 ('сообщения информационная', 0.4577),
 ('сообщений относящихся различным', 0.4576),
 ('из новостного информационного', 0.4522),
 ('просмотре новых сообщений', 0.4516),
 ('новостного информационного', 0.4499),
 ('сообщения которые описывают', 0.4478),
 ('текстовых сообщений относящихся', 0.446),
 ('сообщения описываются', 0.4454),
 ('дружественного оппонента валерия', 0.4432),
 ('сообщения информационная часть', 0.4421)]

In [ ]:
with open('KeyBERT_top_20_trigram.csv', 'w', encoding='utf-8') as f2:
    for eachkwrd in keywords:
        f2.write(str(eachkwrd[0])+";"+str(eachkwrd[1])+"\n")

In [ ]:
#RuTermExtract

In [ ]:
!pip install rutermextract

In [ ]:
from rutermextract import TermExtractor

In [ ]:
term_extractor = TermExtractor()

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
for term in term_extractor(text):
    print(term.normalized, term.count)

сообщение 12
пользователь 8
поток 6
сообщения 5
читатель 4
события 4
алгоритм 4
подобные сообщения 3
входное поток 3
помощь 3
кластер 3
время 3
такая реклама 2
разное время 2
пользовательский кластер 2
одна сторона 2
кластер пользователя 2
база данных 2
адресная реклама 2
событие 2
связь 2
реклама 2
расстояние 2
объекты 2
объект 2
данные 2
r-поток 2
r-пользователь 2
статистический критерий расстояния 1
созданный сами пользователь 1
различные семантические средства 1
новостные сайты запросов 1
новостное информационное поток 1
московское гуманитарное университет 1
ключевые слова запроса 1
каждый новые сообщение 1
информационная часть алгоритма 1
два информационных блоков 1
близкие информационные потребности 1
адресная реклама сети 1
яркий пример 1
ядро алгоритма 1
функции поиска 1
фундаментальная идеи 1
текстовые сообщения 1
такая процедура 1
такая профиль 1
такая база 1
схожие характеристики 1
страница выдачи 1
статистический критерий 1
стандартные грамматик 1
специальная программа 1
со

In [ ]:
with open('RuTermExtract_keywords.csv', 'w', encoding='utf-8') as f2:
    for term in term_extractor(text):
        f2.write(str(term.normalized)+";"+str(term.count)+"\n")

In [ ]:
#SpaCy

In [ ]:
!pip install spacy

In [ ]:
!pip install https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.1.0/ru_core_news_sm-3.1.0.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 51.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [1]:
import spacy
from collections import Counter
from nltk.corpus import stopwords

In [2]:
# Для SpaCy (дополнительно взяли стоп-слова из nltk, с ними результаты улучшились)
spacy_model = spacy.load('ru_core_news_sm')

/usr/local/lib/python3.10/dist-packages/torch/__init__.py:696: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


In [3]:
spacy_model.max_length = 2400000

In [4]:
nltk_stopwords = stopwords.words('russian')

In [5]:
spacy_stopwords = spacy_model.Defaults.stop_words

In [6]:
def extract_keywords_with_spacy(text):
    keywords = []
    doc = spacy_model(text.read())
    for token in doc:
        # Тексты уже очищены от знаков препинания и лемматизированы, тем не менее, остались некоторые символы,
        # которые нужно удалить
        if token.text not in spacy_stopwords and token.text not in nltk_stopwords and \
                token.text not in "`'«»...—-":
            if token.pos_ in ('ADJ', 'NOUN', 'VERB'):
                keywords.append(token.text)

    freq_word = Counter(keywords)
    max_freq = Counter(keywords).most_common(1)[0][1]
    for w in freq_word:
        freq_word[w] = round(freq_word[w] / max_freq, 3)

    freq_20 = freq_word.most_common(20)

    with open('spacykeywords.csv', 'w', encoding='utf-8') as f2:
        for term in freq_20:
            f2.write(str(term[0])+";"+str(term[1])+"\n")

    print(freq_20)

In [7]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as file:
    extract_keywords_with_spacy(file)

[('потока', 1.0), ('сообщения', 0.692), ('пользователя', 0.615), ('сообщений', 0.385), ('является', 0.385), ('реклама', 0.385), ('время', 0.385), ('разное', 0.308), ('данных', 0.308), ('новостных', 0.231), ('текстовых', 0.231), ('Известно', 0.231), ('алгоритм', 0.231), ('кластер', 0.231), ('входного', 0.231), ('стороны', 0.231), ('запросов', 0.231), ('сообщение', 0.231), ('сообщениями', 0.231), ('события', 0.231)]


In [ ]:
#Pullenti

In [ ]:
!pip install pullenti_wrapper

In [ ]:
from pullenti_wrapper.processor import Processor, GEO, ORGANIZATION, PERSON, PHONE, ADDRESS
from collections import Counter

In [ ]:
pullenti_processor = Processor([PERSON, ORGANIZATION, GEO, PHONE, ADDRESS])

In [ ]:
def extract_keywords_with_pullenti(text):
    keywords = []
    for text in text.read().splitlines():
        try:
            result = pullenti_processor(text)
            for match in result.walk():
                for key, value in match.referent.slots:
                    if key not in ('ISRELATIVE', 'SEX', 'NUMBER', 'HIGHER', 'PROFILE',
                                   'ALPHA2', 'POINTER', 'FROM', 'REF', 'ATTRIBUTE'):
                        if 'LASTNAME' in str(match.referent.slots) and 'FIRSTNAME' in str(match.referent.slots):
                            try:
                                keywords.append(f'{match.referent.firstname} {match.referent.lastname}'.lower())
                            except AttributeError:
                                continue
                        keywords.append(str(value).lower())
        except ValueError:
            continue

    freq_word = Counter(keywords)

    freq_20 = freq_word.most_common(20)

    with open('pullenti_keywords.csv', 'w', encoding='utf-8') as f2:
        for term in freq_20:
            f2.write(str(term[0])+";"+str(term[1])+"\n")

    print(freq_20)

In [ ]:
with open('/content/drive/MyDrive/СемАн/Pevzner_IMS_2015_rus.txt', 'r', encoding='utf-8') as file:
    extract_keywords_with_pullenti(file)

[('университет', 2), ('память', 1), ('google', 1), ('гугл', 1), ('социальная сеть', 1), ('facebook', 1), ('фейсбук', 1), ('брауншвейгский университет', 1), ("georeferent(label='geo', slots=[slot(key='name', value='москва'), slot(key='type', value='город')])", 1), ('московский гуманитарный университет', 1)]
